# Implied Volatility Surface Reconstruction
## Finance Club, IIT Roorkee: Open Projects 2026

This notebook implements a robust, deterministic, and mathematically sound **Cross-Sectional Log-Moneyness Linear Extrapolation** algorithm to reconstruct the missing implied volatility surface.

---

### 1. Financial Intuition & Theoretical Foundation

The core of this approach is deeply rooted in options pricing theory, specifically acknowledging the **"Sticky Moneyness"** (or Sticky Delta) dynamics observed in equity index options like the NIFTY 50.

**Why Log-Moneyness ($k = \ln(K/S)$)?**
Implied Volatility fundamentally scales with Moneyness rather than absolute Strike ($K$). If the underlying price ($S$) moves, the entire volatility smile shifts. By transforming the strike prices into Log-Moneyness space, the lowest point of the smile is mathematically centered near $0$ (At-The-Money). Interpolating in this space ensures that the natural curvature and skew of the volatility surface are perfectly preserved, regardless of where the underlying index is trading at a given timestamp.

**Why Cross-Sectional Linear Interpolation?**
Machine Learning algorithms (like Gradient Boosting) often treat financial data as a purely mechanical mapping, ignoring structural laws like Put-Call parity or smile convexity. Furthermore, ML models are highly susceptible to **Leaderboard Shakeups** if the private evaluation data contains a different market regime (e.g., a volatility spike). 

Our algorithm calculates the curve dynamically from scratch for *every single timestamp*. It relies strictly on cross-sectional data points available at that exact moment in time, connecting them with linear splines that trace the discrete smile. This makes the model perfectly immune to temporal distribution shifts.

**Zero Look-Ahead Bias Guarantee**
The competition strictly prohibits look-ahead bias. This model strictly iterates row-by-row. It predicts missing values at timestamp $T$ using ONLY valid observations from timestamp $T$. A final fallback utilizes Forward-Fill (`ffill`), which explicitly carries past data into the present. No future data (`bfill`) is ever utilized.

### 2. Data Loading and Preparation
We begin by loading the dataset and separating the Call (CE) and Put (PE) option columns.

In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import warnings

warnings.filterwarnings('ignore')

print("Loading dataset...")
df_original = pd.read_csv('dataset.csv')
df = df_original.copy()

# Identify option columns and separate by Call/Put
option_cols = [c for c in df_original.columns if c.startswith('NIFTY')]
ce_cols = sorted([c for c in option_cols if c.endswith('CE')])
pe_cols = sorted([c for c in option_cols if c.endswith('PE')])

# Extract numerical strikes from column names
strikes_ce = np.array([float(c.replace('NIFTY27JAN26','').replace('CE','')) for c in ce_cols])
strikes_pe = np.array([float(c.replace('NIFTY27JAN26','').replace('PE','')) for c in pe_cols])

filled = df[option_cols].copy()

### 3. Cross-Sectional Interpolation Engine
Here we define the core engine. For every timestamp (row):
1. We extract the valid strikes and their corresponding IVs.
2. We transform the strikes into Log-Moneyness: $k = \ln(K/S)$.
3. We fit a linear spline (`interp1d`) to the available points.
4. We predict the missing points. For points falling outside the boundaries of available strikes, we use **Linear Slope Extrapolation** (`fill_value='extrapolate'`), which correctly models the continuation of the volatility skew out to the wings.
5. We clip the output to sensible boundaries to prevent mathematical artifacts at extreme wings.

In [ ]:
print("Reconstructing Implied Volatility Surface...")
for ix, row in df.iterrows():
    if pd.isna(row['datetime']): continue
        
    S = row['underlying_price']
    
    # ------------------- CE Reconstruction -------------------
    valid_ce_strikes, valid_ce_ivs = [], []
    for j, c in enumerate(ce_cols):
        val = row[c]
        if pd.notna(val):
            valid_ce_strikes.append(strikes_ce[j])
            valid_ce_ivs.append(val)
            
    if len(valid_ce_strikes) >= 2:
        # Transform to Log-Moneyness Space
        valid_ce_log_m = np.log(np.array(valid_ce_strikes) / S)
        f_ce = interp1d(valid_ce_log_m, valid_ce_ivs, kind='linear', fill_value='extrapolate')
        
        for j, c in enumerate(ce_cols):
            if pd.isna(row[c]):
                log_m = np.log(strikes_ce[j] / S)
                pred = float(f_ce(log_m))
                filled.at[ix, c] = np.clip(pred, 0.01, 6.0)
                
    elif len(valid_ce_strikes) == 1:
        for c in ce_cols:
            if pd.isna(row[c]):
                filled.at[ix, c] = valid_ce_ivs[0]

    # ------------------- PE Reconstruction -------------------
    valid_pe_strikes, valid_pe_ivs = [], []
    for j, c in enumerate(pe_cols):
        val = row[c]
        if pd.notna(val):
            valid_pe_strikes.append(strikes_pe[j])
            valid_pe_ivs.append(val)
            
    if len(valid_pe_strikes) >= 2:
        # Transform to Log-Moneyness Space
        valid_pe_log_m = np.log(np.array(valid_pe_strikes) / S)
        f_pe = interp1d(valid_pe_log_m, valid_pe_ivs, kind='linear', fill_value='extrapolate')
        
        for j, c in enumerate(pe_cols):
            if pd.isna(row[c]):
                log_m = np.log(strikes_pe[j] / S)
                pred = float(f_pe(log_m))
                filled.at[ix, c] = np.clip(pred, 0.01, 6.0)
                
    elif len(valid_pe_strikes) == 1:
        for c in pe_cols:
            if pd.isna(row[c]):
                filled.at[ix, c] = valid_pe_ivs[0]

# Final fallback for completely empty rows (if any) ensuring zero look-ahead bias (only ffill)
for c in option_cols:
    if filled[c].isna().any():
        filled[c] = filled[c].fillna(method='ffill')

# Save fully filled dataset intermediate file
df_filled = df_original.copy()
df_filled[option_cols] = filled
df_filled.to_csv("filled_dataset.csv", index=False)
print("Filled dataset saved as 'filled_dataset.csv'.")

### 4. Generate Official Submission Format
Finally, we extract only the missing data points from our fully populated `filled_dataset.csv` and format them into the exact structure required by the Kaggle judging system.

In [ ]:
ORIGINAL_DATASET_PATH = "dataset.csv" 
SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str = "submission.csv"):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled_df   = pd.read_csv(filled_path)

    feature_cols = [c for c in original.columns if c != "datetime" and c != "underlying_price"]

    rows = []
    for col in feature_cols:
        was_missing = original[col].isna()

        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled_df.loc[idx, col]
            rows.append({"id": uid, "value": val})

    solution = pd.DataFrame(rows, columns=["id", "value"])
    # The official submission requires column name 'implied_volatility', not 'value'
    solution.rename(columns={'value': 'implied_volatility'}, inplace=True)
    solution = solution.sort_values("id").reset_index(drop=True)
    solution.to_csv(output_path, index=False)
    print(f"\u2705 Solution saved \u2192 {output_path}  ({len(solution)} rows)")

generate_solution("filled_dataset.csv", "submission.csv")